# FDS 학습 데이터 그룹 3분할 v2 — 팀 공통 데이터셋 생성
**DGU-P-개발-데이터분할-01 (v2)** · 팀 project · 하이브리드 파이프라인 공통 입력

이 노트북은 `FDS_ML_모델개발_Colab.ipynb` 4단계(그룹 3분할)와 **동일한** 분할을 수행하고,
각 분할을 원본 컬럼 그대로(라벨 포함) parquet 3개로 저장한다.
메인 분류 · 이상탐지 · 시퀀스 딥러닝 세 축이 모두 이 파일을 공통으로 사용한다.

**v2 변경점**

1. 저장 형식 변경: X/y 분리 6파일 → **라벨 포함 단일 파일 3개** (`train.parquet` / `valid.parquet` / `test.parquet`, 맨 앞 `row_id`)
2. 속도 수정: 새 데이터에 고유값 수십만 개짜리 문자열 컬럼(`TRGTER_ID`, `PRHNF_ID`, `DELNG_DT` 등)이 추가되면서
   기존 `upper_categories`의 `lv.index()` 선형 탐색이 O(고유값²)로 폭발 → **dict 조회로 교체(결과 동일, 수 시간 → 수 초)**

**분할 사양** (ML 수행보고서 5.2절 — 원본 100컬럼 파일 기준 기대값)

| split | 방식 | 건수(기대) | 양성률(기대) |
|---|---|---|---|
| train | 그룹 60%, 원비율 유지 | 395,733 | 0.5332 |
| valid | 그룹 20%, 0.083 재표집 | 67,263 | 0.0830 |
| test | 그룹 20%, 0.083 재표집 | 67,227 | 0.0830 |

- 그룹 키: 준식별자 9개 정수 해시(`s*100003+v`) — 같은 '사람'이 두 split에 걸치지 않음
- 여기서의 **test는 입력 파일 내부의 그룹 홀드아웃**이다. 사보원 제공 test.parquet(30,141건)과 무관.

**실행 방법**: 위에서부터 순서대로 실행 → 2단계에서 학습용 parquet 업로드 → 마지막 셀에서 `fds_split_v2.zip` 자동 다운로드


## 1. 준비

In [ ]:
import os, json, gc, time, hashlib, zipfile
from pathlib import Path
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings('ignore')
print('환경 | pandas', pd.__version__, '| numpy', np.__version__)

## 2. 학습용 parquet 업로드

`/content/data/train.parquet` 위치를 사용한다. 이미 올려둔 파일이 있으면 재업로드 없이 그대로 쓴다. (`output (1).parquet` 같은 이름이어도 가장 큰 parquet을 자동으로 train으로 사용)

In [ ]:
from google.colab import files as colab_files

DATA_DIR = Path('/content/data'); DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT      = Path('/content/split_out'); OUT.mkdir(parents=True, exist_ok=True)
TRAIN_PATH = DATA_DIR / 'train.parquet'

if not TRAIN_PATH.exists():
    pool = list(DATA_DIR.glob('*.parquet')) + list(Path('/content').glob('*.parquet'))
    if not pool:
        for name in colab_files.upload():
            os.replace(name, DATA_DIR / Path(name).name)
        pool = list(DATA_DIR.glob('*.parquet')) + list(Path('/content').glob('*.parquet'))
    pool = sorted(pool, key=lambda p: p.stat().st_size, reverse=True)
    if pool[0].resolve() != TRAIN_PATH.resolve():
        os.replace(pool[0], TRAIN_PATH)

print(f'train: {TRAIN_PATH} ({TRAIN_PATH.stat().st_size/1e6:.1f}MB)')

## 3. 전처리 — ML 노트북 3단계와 동일 (분할 키 계산용)

분할 그룹 키가 `upper_categories` 이후의 category 코드에 의존하므로 ML 노트북의 전처리·시점 정렬을 재현한다.
아래 코드는 원본에서 두 곳만 수정했다.

- `upper_categories`: `lv.index()` → dict 조회 (**결과는 완전히 동일**, ID·일시처럼 고유값이 수십만 개인 컬럼에서도 수 초)
- `prepare_train`: 원본 행 번호 복원용 `order` 반환 추가, 단계별 소요시간 출력 추가

In [ ]:
LABEL     = 'INJSTUSE_AT'
ORDER_COL = 'CONFM_NO'
SEED      = 42
np.random.seed(SEED)

DROP_COLS = ['CONFM_NO','DELNG_SN','BSNS_SE_ID','TRGTER_LTR_DPLCT_AT','INORG_HSPTLZ_AT',
             'DELNG_MONTH','DELNG_QUARTER']
AMOUNT_COLS = ['GVSPMN_AM','SFALMT_AM','DDCT_PONT_SM_AM','REMNDR_PONT_AM','CREAT_PONT_AM',
               'CREAT_SFALMT_AM','CREAT_GVSPMN_AM','HLTHINS_CALC_AM','F_AMT_PER_MIN']
UNIT_COL  = 'VOUCH_UNTPC'

# 그룹(사람) 대용 키 — 준식별자 9개 (보고서 분할과 동일 유지)
GROUP_KEYS = ['TRGTER_AGE','TRGTER_SEXDSTN','TRGTER_GRAD_SE','DEPT_CODE','TROBL_TY',
              'MBHSHD_CO','INCOME_SE','PRHNF_AGE','PRHNF_SEXDSTN']


def read_parquet(path):
    df = pd.read_parquet(path)
    for c in df.columns:
        dt = str(df[c].dtype)
        if dt in ('string','large_string','object'):
            df[c] = df[c].astype(str).astype('category')
        elif dt.startswith(('Int','Float','boolean')):
            df[c] = df[c].astype('float64')
    return df


def upper_categories(df):
    for c in [c for c in df.columns if str(df[c].dtype)=='category']:
        s = df[c]
        mp = {x: str(x).strip().upper() for x in s.cat.categories}
        lv = sorted(set(mp.values()))
        pos = {v: i for i, v in enumerate(lv)}      # v2: lv.index() 선형탐색 → dict (결과 동일)
        lut = np.array([pos[mp[x]] for x in s.cat.categories])
        cd = s.cat.codes.to_numpy()
        df[c] = pd.Categorical.from_codes(np.where(cd<0,-1,lut[np.clip(cd,0,None)]), categories=lv)
    return df


def prepare_keys():
    t0=time.time(); df = read_parquet(TRAIN_PATH)
    print(f'로드 {df.shape} ({time.time()-t0:.0f}초)')
    t0=time.time(); df = upper_categories(df)
    print(f'대문자 정규화 완료 ({time.time()-t0:.0f}초)')
    y = df[LABEL].astype(int).to_numpy()
    order = np.argsort(df[ORDER_COL].to_numpy(), kind='stable')
    keep = [c for c in GROUP_KEYS if c in df.columns]
    assert len(keep)==9, f'그룹 키 컬럼 누락: {set(GROUP_KEYS)-set(keep)}'
    K = df[keep].copy()          # 분할에는 그룹 키 9개만 필요
    del df; gc.collect()
    K = K.take(order).reset_index(drop=True); y = y[order]
    print(f'그룹 키 {K.shape[1]}개 | {len(K):,}건 | 양성률 {y.mean():.4f}')
    return K, y, order


K, y, ORDER = prepare_keys()   # order: 시점정렬 위치 i → 원본 행 번호 order[i]

## 4. 그룹 3분할 — ML 노트북 4단계 코드 그대로

In [ ]:
OPERATING_PRIOR = 0.083

def group_key(X):
    gk = [c for c in GROUP_KEYS if c in X.columns]
    s = None
    for c in gk:
        v = (X[c].cat.codes if str(X[c].dtype)=='category'
             else pd.to_numeric(X[c], errors='coerce').fillna(-1)).astype('int64')
        s = v if s is None else s*100003 + v
    return s.to_numpy()

def prior_match(y, idx, rate, seed=SEED):
    '''라벨만 사용해 양성률을 rate로 다운샘플(피처 미참조).'''
    r = np.random.RandomState(seed); yy = y[idx]
    po, ne = idx[yy==1], idx[yy==0]
    if len(po)/max(len(idx),1) > rate:
        k = min(len(po), max(1, int(round(len(ne)*rate/(1-rate))))); po = r.choice(po, k, replace=False)
    else:
        k = min(len(ne), max(1, int(round(len(po)*(1-rate)/rate)))); ne = r.choice(ne, k, replace=False)
    return np.sort(np.concatenate([po, ne]))

G = group_key(K)
_ug = np.unique(G); np.random.RandomState(SEED).shuffle(_ug)
_a, _b = int(len(_ug)*0.6), int(len(_ug)*0.8)
_S = [set(_ug[:_a]), set(_ug[_a:_b]), set(_ug[_b:])]
TR = np.flatnonzero(np.fromiter((x in _S[0] for x in G), bool, len(G)))
VA = prior_match(y, np.flatnonzero(np.fromiter((x in _S[1] for x in G), bool, len(G))), OPERATING_PRIOR)
EV = prior_match(y, np.flatnonzero(np.fromiter((x in _S[2] for x in G), bool, len(G))), OPERATING_PRIOR)

_ag = pd.DataFrame({'g':G,'y':y}).groupby('g').y.agg(['mean','size']); _m = _ag[_ag['size']>=2]
print(f'그룹 {len(_ag):,} | 2건 이상 그룹 라벨 완전일치 '
      f'{((_m["mean"]==0)|(_m["mean"]==1)).mean():.1%} ← 그룹 분할 필요성 근거')
print(f'겹침 검사: 학습∩검증 {len(set(G[TR])&set(G[VA]))} 학습∩평가 {len(set(G[TR])&set(G[EV]))}')
print(pd.DataFrame([('train',len(TR),y[TR].mean()),('valid',len(VA),y[VA].mean()),
                    ('test',len(EV),y[EV].mean())],
                   columns=['split','n','양성률']).round(4).to_string(index=False))

## 5. 재현 검증

ML 수행보고서 기대값과 비교한다. **불일치가 나오면** 입력 파일이 원본 100컬럼 train.parquet과 내용이 다른 새 버전이라는 뜻이다
(새 데이터에 ID 컬럼이 추가돼도 그룹 키 9개와 CONFM_NO가 같으면 일치해야 정상).
불일치 시 아래 `STRICT = False`로 바꾸면 경고만 남기고 계속 진행한다 — 단, 이 경우 보고서 분할과 다른 분할이므로 반드시 개발팀(PL)와 협의.

In [ ]:
STRICT = True   # 기대값 불일치 시 중단할지 여부

EXPECTED = {'train': (395733, 0.5332), 'valid': (67263, 0.0830), 'test': (67227, 0.0830)}
actual   = {'train': (len(TR), round(float(y[TR].mean()),4)),
            'valid': (len(VA), round(float(y[VA].mean()),4)),
            'test' : (len(EV), round(float(y[EV].mean()),4))}

ok = True
for k in EXPECTED:
    match = EXPECTED[k] == actual[k]
    ok &= match
    print(f'{k:6s} 기대 {EXPECTED[k]} | 실제 {actual[k]} | {"OK" if match else "불일치!"}')

assert len(set(G[TR]) & set(G[VA])) == 0 and len(set(G[TR]) & set(G[EV])) == 0 \
       and len(set(G[VA]) & set(G[EV])) == 0, '그룹 겹침 발견 — 분할 실패'

def split_md5(idx):
    ids = np.sort(ORDER[idx]).astype('int64')
    return hashlib.md5(ids.tobytes()).hexdigest()[:12]

CHECKSUMS = {k: split_md5(v) for k, v in [('train',TR),('valid',VA),('test',EV)]}
print('\n체크섬(원본 행번호 기준):', CHECKSUMS)

if ok:
    print('\n재현 검증 통과 — ML 노트북 분할과 동일')
elif STRICT:
    raise AssertionError('건수/양성률이 보고서와 불일치 — 입력 파일 버전 확인 후 PL과 협의 (계속하려면 STRICT=False)')
else:
    print('\n경고: 보고서 분할과 다름. 새 데이터 기준 분할로 간주하고 계속 진행함 — PL 협의 필수')

## 6. 분할 데이터 저장 — 라벨 포함 단일 파일 3개

- `train.parquet` / `valid.parquet` / `test.parquet`: **원본 컬럼 전부(라벨 INJSTUSE_AT 포함)** + 맨 앞 `row_id`
- `row_id` = 입력 parquet의 원본 행 번호(0부터). 세 축 산출물 결합 시 조인 키
- 주의: 라벨이 파일 안에 있으므로 **학습 시 피처에서 `INJSTUSE_AT`(와 `row_id`)를 반드시 제외**할 것

In [ ]:
raw = pd.read_parquet(TRAIN_PATH)   # dtype 변환 없이 원본 그대로
assert len(raw) == len(K), '원본 재로드 건수 불일치'

# 원본 좌표계 라벨 벡터 (교차 검증용): y는 시점정렬 순서이므로 되돌린다
y_orig = np.empty(len(y), dtype='int64'); y_orig[ORDER] = y

manifest = {'version': 'v2', 'seed': SEED, 'operating_prior': OPERATING_PRIOR,
            'group_keys': GROUP_KEYS, 'source_rows': int(len(raw)),
            'source_cols': int(raw.shape[1]), 'checksums': CHECKSUMS, 'splits': {}}

for name, idx in [('train', TR), ('valid', VA), ('test', EV)]:
    ids = np.sort(ORDER[idx])                     # 원본 행 번호, 오름차순
    part = raw.take(ids).reset_index(drop=True)
    ycheck = pd.to_numeric(part[LABEL], errors='raise').astype('int64')
    assert (ycheck.to_numpy() == y_orig[ids]).all(), f'{name}: 라벨 정합성 불일치'
    part.insert(0, 'row_id', ids.astype('int64'))
    part.to_parquet(OUT / f'{name}.parquet', index=False)
    manifest['splits'][name] = {'n': int(len(ids)), 'pos_rate': round(float(ycheck.mean()), 4),
                                'cols': int(part.shape[1])}
    print(f'{name:6s} {part.shape} 양성률 {ycheck.mean():.4f} 저장 완료')

with open(OUT / 'split_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
del raw; gc.collect()
print('\n저장 위치:', OUT)

## 7. zip으로 묶어 다운로드

In [ ]:
zpath = Path('/content/fds_split_v2.zip')
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        z.write(p, p.name)
print(f'{zpath} ({zpath.stat().st_size/1e6:.1f}MB)')
colab_files.download(str(zpath))

## 8. 팀 사용 가이드 (공유용)

**공통 규약 — 세 축 모두 동일하게 적용**

1. **모델 적합(fit)은 `train.parquet`으로만** 한다. 스케일러·인코더·임계값·그래프 등 데이터에서 학습되는 모든 것 포함.
2. `valid.parquet`은 하이퍼파라미터 튜닝, threshold 탐색, early stopping, 결합 가중치 탐색에 사용.
3. `test.parquet`은 **최종 확정 후 마지막 1회만** 평가. test를 보고 무엇이든 고치는 순간 독립 평가셋 자격을 잃는다.
4. **`INJSTUSE_AT`(라벨)과 `row_id`는 절대 피처로 넣지 말 것.** 컬럼 자동 선택 코드를 쓸 때 특히 주의 — 라벨이 들어가면 F1이 1.0 근처로 나오며 그 결과는 전부 무효.
5. 무작위 재분할 금지 — 이 분할은 사람(그룹) 단위라서 행 단위로 다시 섞으면 그룹 누수로 점수가 부풀려진다(실측 F1 +0.15).
6. 산출물 형식은 ML 보고서 12.3 결합 규약: `row_id, y, score_*` CSV, score는 0~1(확률 아니면 `rank(pct=True)`).

**축별 참고**

- **이상탐지(AE·iForest)**: 학습은 `INJSTUSE_AT==0`인 정상 행만 필터해서 사용(계획서 6.5.3). train 양성률 53%는 무관. threshold·정규화는 valid에서.
  단, 이 train의 '정상'은 1% 언더샘플 선별셋이라 자연 분포와 다름 — 결과 해석 시 명시.
- **시퀀스(RNN/Transformer)**: 시점 정렬은 `CONFM_NO`(또는 새 데이터의 `DELNG_DT`) 기준. 새 데이터에 `TRGTER_ID`/`PRHNF_ID`가 있으면 **실제 ID 기준으로 이용자별 시퀀스 구성 가능**. 시퀀스도 train 행만으로 구성 원칙.
- **관계 네트워크(GNN 피처)**: 새 데이터의 `TRGTER_ID`–`PRHNF_ID`로 실제 관계 그래프 구성 가능. 그래프·노드 피처는 train만으로 적합, valid/test는 조회만(신규 노드는 기본값 + 신규 플래그).
- **메인 분류(LightGBM)**: ML 노트북 전처리가 코드에 내장. `row_id`·라벨 제외 후 입력.

**주의**: 여기서의 test는 입력 파일 내부 그룹 홀드아웃이다. 사보원 제공 test.parquet(2026 Q1, 30,141건)은 별개.